<a href="https://colab.research.google.com/github/yash-2580-ai-ml/Machine-learning-lab-programs/blob/main/naive_bayes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [14]:
import numpy as np

class NaiveBayes:
    def __init__(self):
        # Dictionary to store the mean, variance, and prior probability
        self.class_data = {}
        self.classes = []

    def fit(self, X, y):
        # Find the unique classes (e.g., 0 for malignant, 1 for benign)
        self.classes = np.unique(y)
        total_samples = len(X)

        for c in self.classes:
            # Filter the dataset for the current class
            X_class = X[y == c]

            # Calculate mean and variance for every feature (column)
            class_mean = np.mean(X_class, axis=0)
            class_var = np.var(X_class, axis=0)

            # Prior probability: (rows in this class) / (total rows)
            class_prior = len(X_class) / total_samples

            # Store everything to use later during prediction
            self.class_data[c] = {
                'mean': class_mean,
                'var': class_var,
                'prior': class_prior
            }

    def _calculate_gaussian_probability(self, x, mean, var):
        # Standard math formula for a normal distribution (bell curve)
        # Adding 1e-6 to variance so we never accidentally divide by zero
        exponent = np.exp(-((x - mean) ** 2) / (2 * (var + 1e-6)))
        return (1 / np.sqrt(2 * np.pi * (var + 1e-6))) * exponent

    def predict(self, X):
        predictions = []

        # Predict one row at a time
        for row in X:
            pred = self._predict_single_row(row)
            predictions.append(pred)

        return np.array(predictions)

    def _predict_single_row(self, row):
        best_class = None
        highest_probability = -float('inf')

        for c in self.classes:
            data = self.class_data[c]

            # Start with log of prior probability to prevent numbers from shrinking to 0
            probability = np.log(data['prior'])

            # Add log probability for every feature in this specific row
            for i in range(len(row)):
                mean = data['mean'][i]
                var = data['var'][i]

                feature_prob = self._calculate_gaussian_probability(row[i], mean, var)

                # 1e-9 prevents log(0) errors if the probability is basically zero
                probability += np.log(feature_prob + 1e-9)

            # Update if this class has a higher score
            if probability > highest_probability:
                highest_probability = probability
                best_class = c

        return best_class


# ==========================================
# Testing the model on a realistic dataset
# ==========================================
if __name__ == "__main__":
    from sklearn.datasets import load_breast_cancer
    from sklearn.model_selection import train_test_split

    # Using Breast Cancer dataset because it is larger and harder than Iris
    # It has 569 samples and 30 features, so it won't hit a fake 100% accuracy.
    dataset = load_breast_cancer()
    X, y = dataset.data, dataset.target

    # Split into 80% training and 20% testing
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    model = NaiveBayes()
    model.fit(X_train, y_train)

    predictions = model.predict(X_test)

    # Manually calculate accuracy
    correct = 0
    for i in range(len(y_test)):
        if predictions[i] == y_test[i]:
            correct += 1

    accuracy = (correct / len(y_test)) * 100

    print("--- Naive Bayes Classifier ---")
    print(f"Dataset: Breast Cancer Wisconsin")
    print(f"Total test samples: {len(y_test)}")
    print(f"Correctly predicted: {correct}")
    print(f"Misclassified: {len(y_test) - correct}")
    print(f"Realistic Accuracy: {accuracy:.2f}%")

--- Naive Bayes Classifier ---
Dataset: Breast Cancer Wisconsin
Total test samples: 114
Correctly predicted: 110
Misclassified: 4
Realistic Accuracy: 96.49%
